# An End-to-End AI Harvest Planner for Low-Cost Fruit-Picking Robots Built on a Physics-Consistent World Model

**`04_fidelity.ipynb`**

Every figure in `03_evaluation` rests on one thing: the outcome model's probabilities. The
planner ranks positions by the expected utility they imply, the pick policy reads the same
posterior, and the shift totals are drawn from it. If those probabilities are optimistic, so is
everything above them, and no amount of paired seeds would show it — the comparison would be
consistently wrong rather than noisily wrong.

So the plan is run again, closed loop, against the physics itself. The planner chooses, the
policy picks, and MuJoCo decides what actually happens; the model is not consulted for the
outcome. What is compared is the success rate the model predicted for those picks against the
success rate the physics delivered.

**This is not sim-to-real.** The physics here is the same simulator the model was fitted on, so
agreement is internal consistency, not evidence about an orchard. The name for what is measured
is surrogate fidelity: whether a model trained on single picks still describes the world when a
planner strings hundreds of them together and the canopy thins as it goes.

**The configuration is the one that ships**, taken from `src/planner.py`: twenty stops from the
trained planner, a sweep stage for whatever those twenty miss, and no selection threshold. An
earlier version of this notebook ran three stops at a 0.9 threshold, which reached a third of what
the arm can touch; those figures are not comparable to these and are not carried forward.

**Stated before the run.** The last measurement, on the resampled canopy with the old
interpolated pose, put predicted 0.975 against realised 0.841 — a gap of 0.134, most of it
traceable to a deliberately denser canopy than the detections. The measured pose and the
retrained weights should not widen that. Nor should the wider plan: it attempts every fruit the stops reach, including the ones the old threshold left alone, and those are the harder ones. A gap materially larger than 0.134 means something in
the chain changed for the worse and the evaluation numbers need re-reading before they are
reported.



In [1]:
import json, os, sys, time, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mujoco

# Default to the folder this notebook sits in, so a clone runs without setup. An absolute
# default only ever pointed at one machine, and an environment variable set in a shell does
# not reach a kernel that was already running.
ROOT = Path(os.environ.get("AIPICK_ROOT") or Path.cwd())
os.environ["AIPICK_ROOT"] = str(ROOT)
SRC, DATA, MODELS = ROOT/"src", ROOT/"data", ROOT/"models"
OUT  = ROOT/"runs"/"fidelity"; OUT.mkdir(parents=True, exist_ok=True)
LOGS = OUT/"logs"; LOGS.mkdir(exist_ok=True)
sys.path.insert(0, str(SRC))

import environment as E
import physics as A

TREES_USED = "trees_measured_pose.csv"
E.load(ROOT, trees=TREES_USED, dynamics=True)
T = E.T

import importlib.util
spec = importlib.util.spec_from_file_location("gen", SRC/"generate.py")
G = importlib.util.module_from_spec(spec); G.__name__ = "gen"
spec.loader.exec_module(G)

HALF_X, LIFT = 0.200, 0.600            # the compact arm, the design target
THRESHOLD, K_STATIONS = 0.9, 3
TREES_EVAL = list(range(20, 50))       # the held-out trees section 6 of 03 used

_v = T[["sdir_x", "sdir_y", "sdir_z"]].to_numpy(float)
_tilt = np.degrees(np.arccos(np.clip(np.abs(_v[:, 1]/np.linalg.norm(_v, axis=1)), 0, 1)))
print(f"canopy {TREES_USED}: stalk tilt median {np.median(_tilt):.2f} deg "
      f"(detections 27.00), measured lean mean {T.lean_deg.mean():.3f} deg")
print(f"dynamics {'on' if E.DYN else 'OFF'}")

pd.set_option("display.width", 200)
print(f"\nmujoco {mujoco.__version__}   torch {torch.__version__}")
print(f"trees {T.tree.nunique()}   classes {E.CLASSES}")
print(f"OBSERVE_SETTLE {G.OBSERVE_SETTLE} steps @ {A.SIM_HZ} Hz "
      f"= {G.OBSERVE_SETTLE/A.SIM_HZ:.1f} s of settling before every pick")

# What the previous measurement found, so the comparison is on the page.
PRIOR = dict(predicted=0.975, realised=0.841, gap=0.134,
             canopy="measured pose, three stops, threshold 0.9")




canopy trees_measured_pose.csv: stalk tilt median 26.64 deg (detections 27.00), measured lean mean 1.455 deg
dynamics on

mujoco 3.11.0   torch 2.9.1+cpu
trees 50   classes ['APPROACH_BLOCKED', 'DEFECT', 'GRASP_FAILED', 'NEIGHBOR_KNOCKED', 'NO_DETACH', 'SUCCESS']
OBSERVE_SETTLE 2400 steps @ 240 Hz = 10.0 s of settling before every pick


## 1. The two pickers, loaded from disk

Nothing is trained here. The three weights files already exist; this notebook reads them.


In [2]:
import planner as PL

info = PL.load(ROOT)
print(f"planner {info['planner']}   policy {info['policy']}   scaling {info['scaling']}")
print(f"  move_secs {info['move_secs'][0]:.3f} / {info['move_secs'][1]:.3f}"
      f"   (training recorded 385.324 / 485.485)")
print(f"configuration: {PL.K_STATIONS} stops + sweep, threshold {PL.THRESHOLD}, "
      f"arm {PL.HALF_X} x {PL.LIFT} m")

# The physics sections below reach into the planner's internals rather than calling plan_tree:
# they have to interleave a MuJoCo pick between one decision and the next, which the planning
# function does not expose. These are the same objects, not copies.
PLANNER, POLICY = PL.PLANNER, PL.POLICY
FMU, FSD = PL.FMU, PL.FSD
HALF_X, LIFT = PL.HALF_X, PL.LIFT
THRESHOLD, K_STATIONS = PL.THRESHOLD, PL.K_STATIONS
prune_positions = PL.prune_positions
policy_features = PL.policy_features
planner_features = PL.planner_features


def tree_case(tid):
    return PL.tree_case(tid)


def decode_stations(case, k=None, chooser="planner", sweep=None):
    """Stops for one tree, from the module -- twenty from the planner plus the sweep stage."""
    S, _ = PL.stations(case, k or PL.K_STATIONS, chooser,
                       PL.SWEEP if sweep is None else sweep)
    return S


CASE_CACHE = {t: tree_case(t) for t in TREES_EVAL}
print(f"\n{len(CASE_CACHE)} trees prepared")
n = np.mean([len(decode_stations(c)) for c in list(CASE_CACHE.values())[:5]])
print(f"stops per tree, first five: {n:.1f}")


c:\python\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


planner station_planner_k20.pt   policy pick_policy.pt   scaling recomputed
  move_secs 385.324 / 485.485   (training recorded 385.324 / 485.485)
configuration: 20 stops + sweep, threshold 0.0, arm 0.2 x 0.6 m

30 trees prepared
stops per tree, first five: 21.0


## 2. The step hook

`harvest_once` exposes `on_step` only to `ContactWatch`, and that watch is `None` when the target
has no neighbours — so there is no hook to hang a pose log on for isolated fruit.

`aipick_mj3` calls `mujoco.mj_step(m, d)` by module attribute at every stepping site, so replacing
that attribute reaches every step in both modules without editing either file. The cell below
checks that the replacement actually fires before anything long is run; if it silently did not,
the pose logs would come out empty after the full run had finished.


In [3]:
_MJ_STEP = mujoco.mj_step


class StepHook:
    '''Counts steps and, when armed, records poses in the TREE frame.

    Scene frame is target-centred with z up; the tree frame has y up. `to_tree` undoes both.
    '''
    R = np.array([[1., 0, 0], [0, 0, 1], [0, 1, 0]])   # symmetric: its own inverse

    def __init__(self):
        self.n = 0
        self.total = 0
        self.armed = False
        self.every = 64           # 240 Hz / 64; a pick runs ~25k steps, so 8 gave a
                                  # 104-second clip per pick at 30 fps. 64 keeps it usable.
        self.rows = []
        self.ctx = None           # dict(centre, names, t0, fruit_ids)
        self.settled = None       # pose of the target at the end of the settle phase

    def to_tree(self, v):
        return self.R @ np.asarray(v, float) + self.ctx["centre"]

    def hook(self, m, d, nstep=1):
        _MJ_STEP(m, d, nstep)
        self.n += nstep; self.total += nstep
        if self.ctx is not None and self.settled is None and self.n >= G.OBSERVE_SETTLE:
            self.settled = self._fruit_pose(m, d)
        if self.armed and self.ctx is not None and self.n % self.every < nstep:
            self.rows.append(self._frame(m, d))

    def _fruit_pose(self, m, d):
        c = np.array(d.body("fruit").xpos, float)
        top = np.array(d.body("seg4").xpos, float)
        ax = (top - c)/max(np.linalg.norm(top - c), 1e-9)
        return dict(lean_deg=float(np.degrees(np.arccos(np.clip(ax[2], -1, 1)))),
                    quat=[float(x) for x in d.body("fruit").xquat])

    def _frame(self, m, d):
        fr = {}
        for body, fid in self.ctx["names"].items():
            try:
                b = d.body(body)
            except Exception:
                continue
            fr[str(fid)] = [round(float(x), 5) for x in self.to_tree(b.xpos)] + \
                           [round(float(x), 5) for x in b.xquat]
        palm = d.body("palm")
        fj = []
        for j in range(3):
            try:
                fj.append(round(float(d.joint(f"fj{j}").qpos[0]), 4))
            except Exception:
                fj.append(0.0)
        return dict(t=round(self.n/A.SIM_HZ, 3),
                    palm=[round(float(x), 5) for x in self.to_tree(palm.xpos)] +
                         [round(float(x), 5) for x in palm.xquat],
                    fj=fj, fruit=fr)

    def begin(self, centre, names, armed):
        self.n = 0; self.settled = None; self.armed = armed
        self.ctx = dict(centre=np.asarray(centre, float), names=names)

    def end(self):
        out, self.rows = self.rows, []
        self.ctx = None; self.armed = False
        return out


HOOK = StepHook()
mujoco.mj_step = HOOK.hook

_before = HOOK.total
_m = mujoco.MjModel.from_xml_string(
    "<mujoco><worldbody><body><freejoint/><geom size='0.01'/></body></worldbody></mujoco>")
_d = mujoco.MjData(_m)
for _ in range(5):
    mujoco.mj_step(_m, _d)
assert HOOK.total == _before + 5, "the mj_step patch did not take -- pose logs would be empty"
print(f"step hook live ({HOOK.total} steps seen), logging every {HOOK.every} steps "
      f"= {A.SIM_HZ/HOOK.every:.0f} fps")


step hook live (5 steps seen), logging every 64 steps = 4 fps


## 3. One pick, in the physics

The same scene the dataset was generated from: the target at the origin plus its two nearest
surviving neighbours within four diameters, its own aperture, its own leaves.

Two departures from `aipick_tree_harvest`, both deliberate:

**`settled_lean_of` is gone.** It compiled a second scene and settled it another 2,400 steps for
every propped fruit, only to fill `obs_lean_deg` on a recording row. Here the decision has already
been made by the planner and the policy, so that observation decides nothing — and `harvest_once`
settles the same scene anyway. The step hook reads the settled pose out of that settle for free.
Propped picks now cost roughly half what they did.

**Latents and the leaf seed stay keyed to `fruit_id`,** so a given fruit meets the same physics
under both pickers. That is what makes the comparison paired at the fruit level, not just the tree
level.

`scene_for` writes `A.PALM_H` and `A.FINGER_CLOSE` as module globals, so a scene must be run
immediately after it is compiled — never compile a batch and run them later.


In [4]:
NEIGHBOUR_MAX_D = 4.0
MAX_NEIGHBOURS  = 2
TOUCH_RATIO     = 1.20
R_TREE2SCENE    = np.array([[1., 0, 0], [0, 0, 1], [0, 1, 0]])


def tree_units(tree_df, fruit_id, taken=frozenset()):
    live = tree_df[~tree_df.fruit_id.isin(taken)]
    tgt = live[live.fruit_id == fruit_id]
    if not len(tgt):
        return None
    tgt = tgt.iloc[0]
    c = np.array([tgt.x, tgt.y, tgt.z], float)
    others = live[live.fruit_id != fruit_id]
    if len(others):
        P = others[["x", "y", "z"]].to_numpy(float)
        d = np.linalg.norm(P - c, axis=1)
        keep = d <= NEIGHBOUR_MAX_D*tgt.dim
        others, d = others[keep], d[keep]
        others = others.iloc[np.argsort(d)[:MAX_NEIGHBOURS]]
    units = [dict(name="A", abs_pt=np.zeros(3),
                  sdir=np.array([tgt.sdir_x, tgt.sdir_y, tgt.sdir_z], float),
                  dim=float(tgt.dim), apple_id=int(tgt.fruit_id))]
    for i, (_, o) in enumerate(others.iterrows()):
        units.append(dict(name=f"N{i}",
                          abs_pt=np.array([o.x, o.y, o.z], float) - c,
                          sdir=np.array([o.sdir_x, o.sdir_y, o.sdir_z], float),
                          dim=float(o.dim), apple_id=int(o.fruit_id)))
    return units, c


def to_scene_frame(units):
    out = []
    for x in units:
        y = dict(x)
        y["abs_pt"] = R_TREE2SCENE @ np.asarray(x["abs_pt"], float)
        y["sdir"]   = R_TREE2SCENE @ np.asarray(x["sdir"], float)
        out.append(y)
    return out


def touching_pairs(tree_df, taken=frozenset()):
    live = tree_df[~tree_df.fruit_id.isin(taken)]
    P = live[["x", "y", "z"]].to_numpy(float)
    D = live.dim.to_numpy(float)
    ids = live.fruit_id.to_numpy()
    out = {}
    for i in range(len(live)):
        d = np.linalg.norm(P - P[i], axis=1); d[i] = np.inf
        j = int(np.argmin(d))
        if d[j] <= TOUCH_RATIO*(D[i] + D[j])/2:
            out[int(ids[i])] = int(ids[j])
    return out


def mj_pick(tree_df, fruit_id, taken=frozenset(), log_poses=False):
    '''One physical pick. Returns the generator's row plus the settled pose and the units.'''
    got = tree_units(tree_df, fruit_id, taken)
    if got is None:
        return None
    units_tree, centre = got
    units = to_scene_frame(units_tree)
    dim = units[0]["dim"]
    ap  = G.aperture_from(1.0, dim)

    m = mujoco.MjModel.from_xml_string(
        G.scene_for(units, ap, dim_true=dim, leaf_seed=fruit_id))
    d = mujoco.MjData(m)
    G.apply_latents(m, G.draw_latents(np.random.default_rng(int(fruit_id))))

    names = {"fruit": int(fruit_id)}
    names.update({f"N{i}_fruit": int(u["apple_id"]) for i, u in enumerate(units[1:])})
    HOOK.begin(centre, names, armed=log_poses)
    row = G.harvest_once(m, d, units, ap, row=pd.Series(dict(apple_id=fruit_id, dim=dim)))
    settled, poses = HOOK.settled, HOOK.end()

    row = dict(row)
    row["fruit_id"] = int(fruit_id)
    row["cls"] = G.to_model_class(row["code"])
    row["mj_lean_deg"] = None if settled is None else round(settled["lean_deg"], 3)
    row["n_units"] = len(units)
    return dict(row=row, poses=poses, units=units_tree, centre=centre, aperture=float(ap))


In [5]:
one = T[T.tree == TREES_EVAL[0]]
fid = int(one[one.vis_any].fruit_id.iloc[0])

t0 = time.time()
r = mj_pick(one, fid, log_poses=True)
dt = time.time() - t0

print(f"fruit {fid}: {r['row']['code']} -> class {r['row']['cls']}   "
      f"{dt:.1f} s, {r['row']['n_units']} bodies, lean {r['row']['mj_lean_deg']} deg")
print(f"pose frames captured: {len(r['poses'])}  "
      f"({len(r['poses'])/max(dt,1e-9):.0f} per wall second)")

t0 = time.time()
_ = mujoco.MjModel.from_xml_string(
    G.scene_for(to_scene_frame(tree_units(one, fid)[0]), G.aperture_from(1.0, 0.075),
                dim_true=0.075, leaf_seed=fid))
print(f"\nXML compile: {(time.time()-t0)*1000:.1f} ms -- "
      f"{(time.time()-t0)/max(dt,1e-9)*100:.2f}% of one pick")


fruit 0: APPROACH_BLOCKED -> class APPROACH_BLOCKED   7.7 s, 3 bodies, lean 4.452 deg
pose frames captured: 392  (51 per wall second)

XML compile: 14.8 ms -- 0.19% of one pick


## 4. One tree, closed loop

The picker is asked again after every physical outcome, so it plans against what actually
happened rather than against a sampled draw. Two bookkeeping sets are kept apart and they are not
the same thing:

```
st.alive   what the picker may still choose      -- a fruit is dropped after ONE attempt
taken      what is physically off the tree       -- only detachments
```

A `GRASP_FAILED` fruit therefore stays in the scene and keeps propping its neighbours, but is
never attempted again. That matches the shift simulation, which removes a fruit from
consideration whatever the outcome, and it matches the limitation already declared in the
manuscript: retries are out of scope. Scoring it any other way would compare a one-shot policy
against a retrying one.

The station a pick is taken from decides the viewpoint, so `vis_*`/`vfrac_*` come from the right
column rather than always from `front`.


In [6]:
def view_of_x(x):
    return min(E.STATION_X, key=lambda s: abs(E.STATION_X[s] - x))


def run_tree_mj(tid, picker="rule", station_fn=decode_stations, max_picks=None,
                log_poses=False, threshold=None):
    """One tree, planned by the model and carried out by the physics.

    `picker` names how the next fruit is chosen. `"rule"` is the shipping configuration -- the
    rate rule with no selection threshold, taking everything the stops reach. `"threshold"` is
    the same rule at 0.9, which leaves what the outcome model is unsure about. `"policy"` is the
    trained pick policy. The first is what the report describes; the other two are the operating
    points it is measured against.
    """
    use_policy = picker == "policy"
    thr = THRESHOLD if threshold is None else threshold
    if picker == "threshold" and threshold is None:
        thr = 0.9
    case = CASE_CACHE[tid]
    geom, Pp, Cp = case["geom"], case["POS"], case["COV"]
    tree_df = T[T.tree == tid]
    id2row = {int(f): i for i, f in enumerate(geom.ids)}

    S = sorted(station_fn(case), key=lambda r: Pp[r][0])
    plan_cov = Cp[S].any(axis=0)
    st = E.TreeState(geom)
    taken, pos, secs, log, poses = set(), None, 0.0, [], []

    while True:
        if max_picks is not None and len(log) >= max_picks:
            break
        live = plan_cov & st.alive
        if not live.any():
            break

        if use_policy:
            ctx = torch.tensor([0.0 if pos is None else pos[0],
                                0.0 if pos is None else pos[1]/4.0,
                                HALF_X, LIFT/2.0, 1.0, st.alive.mean()], dtype=torch.float32)
            X = torch.as_tensor((policy_features(st, Pp, Cp, pos) - FMU)/FSD, dtype=torch.float32)
            with torch.no_grad():
                sc = POLICY.scores(X, live.copy(), ctx)
            a = int(sc.argmax())
            if a == geom.n:
                break
            j = a
        else:
            u = st.utilities()
            ok = live & (u >= thr) if thr > 0 else live
            if not ok.any():
                break
            j = int(np.where(ok, u, -np.inf).argmax())

        reach = [r for r in S if Cp[r, j]]
        p = min((tuple(Pp[r]) for r in reach), key=lambda q: E.move_seconds(pos, q))
        move = E.move_seconds(pos, p)
        fid = int(geom.ids[j])

        pr = st.probabilities(j)
        p_succ = float(pr[E.CLASSES.index("SUCCESS")])
        exp_u  = float(pr @ E.UTIL_VEC)
        env_lean = float(geom.observe_all(st.alive, rows=[j])[0, geom.COL["obs_lean_deg"]])
        nb_rows = geom.disturbed_by(j)                 # who env will re-observe when j goes
        nb_proba, nb_dirty = st.proba[nb_rows].copy(), st.dirty[nb_rows].copy()
        props  = touching_pairs(tree_df, taken)
        st_idx = next(i for i, r in enumerate(S) if tuple(Pp[r]) == p)

        want_poses = log_poses and len(log) < POSE_PICK_CAP
        got = mj_pick(tree_df, fid, taken, log_poses=want_poses)
        if got is None:                       # already gone; should not happen, guard anyway
            st.remove(j); continue
        row, cls = got["row"], got["row"]["cls"]

        st.remove(j)                          # one attempt per fruit, whatever happened
        detached = cls in ("SUCCESS", "DEFECT", "NEIGHBOR_KNOCKED")
        if detached:
            taken.add(fid)
        else:
            # GRASP_FAILED, APPROACH_BLOCKED and NO_DETACH leave the fruit hanging. env has
            # just been told it is gone and would re-observe its neighbours as if a body had
            # vanished, but the physics scene still contains it and it still props them.
            # Roll those neighbours back: the picker drops this fruit, the scene does not.
            st.proba[nb_rows], st.dirty[nb_rows] = nb_proba, nb_dirty
        knocked = None
        if cls == "NEIGHBOR_KNOCKED":
            nb = props.get(fid)
            if nb is not None and nb not in taken:
                taken.add(nb); knocked = int(nb)
                if nb in id2row:
                    st.remove(id2row[nb])

        secs += move + E.PICK_SECONDS
        pos = p
        if want_poses:
            poses.extend([dict(g, step=len(log)+1, station=st_idx) for g in got["poses"]])
        log.append(dict(
            tree=tid, picker=picker, step=len(log)+1,
            fruit=fid, x=round(p[0], 2), mast=round(p[1], 2), view=view_of_x(p[0]),
            propped=fid in props, p_success=round(p_succ, 4), exp_utility=round(exp_u, 4),
            code=row["code"], cls=cls, knocked=knocked,
            mj_lean_deg=row["mj_lean_deg"], env_lean_deg=round(env_lean, 3),
            n_fingers=row["n_fingers"], seat_x_palm=row["seat_x_palm"],
            clearance_mm=row["diag_clearance_mm"],
            neighbour_moved_mm=row["neighbour_moved_mm"],
            secs=round(secs, 1)))

    L = pd.DataFrame(log)
    if len(L):
        premium = int(L.cls.isin(["SUCCESS", "NEIGHBOR_KNOCKED"]).sum())
        summary = dict(tree=tid, picker=picker, threshold=thr,
                       premium=premium, attempts=len(L),
                       lost=int(L.knocked.notna().sum()),
                       success=int((L.cls == "SUCCESS").sum()),
                       knocked=int((L.cls == "NEIGHBOR_KNOCKED").sum()),
                       blocked=int((L.cls == "APPROACH_BLOCKED").sum()),
                       grasp_failed=int((L.cls == "GRASP_FAILED").sum()),
                       defect=int((L.cls == "DEFECT").sum()),
                       predicted=float(L.exp_utility.sum()),
                       seconds=float(L.secs.iloc[-1]),
                       stations=len(S), touching=int(L.propped.sum()))
    else:
        summary = dict(tree=tid, picker=picker, threshold=thr,
                       premium=0, attempts=0, lost=0, success=0, knocked=0, blocked=0,
                       grasp_failed=0, defect=0, predicted=0.0, seconds=0.0,
                       stations=len(S), touching=0)
    return L, summary, poses





In [7]:
POSE_PICK_CAP = 0          # no pose logging during the smoke test

t0 = time.time()
Ls, Ss, _ = run_tree_mj(TREES_EVAL[0], picker="rule", max_picks=4)
print(f"4 picks in {(time.time()-t0)/60:.2f} min "
      f"({(time.time()-t0)/4:.1f} s each)\n")
print(Ls[["step", "fruit", "view", "propped", "p_success", "code", "cls",
          "mj_lean_deg", "env_lean_deg"]].to_string(index=False))
print("\nprojected cost of the full run:")
per = (time.time()-t0)/4
print(f"  {per:.1f} s/pick x ~19 picks x 2 pickers x {len(TREES_EVAL)} trees "
      f"= {per*19*2*len(TREES_EVAL)/60:.0f} min")


4 picks in 0.26 min (3.9 s each)

 step  fruit  view  propped  p_success    code     cls  mj_lean_deg  env_lean_deg
    1     39 right    False     0.9964 SUCCESS SUCCESS          0.0           0.0
    2     66  left    False     0.9959 SUCCESS SUCCESS          0.0           0.0
    3     90 right    False     0.9959 SUCCESS SUCCESS          0.0           0.0
    4    101 front    False     0.9957 SUCCESS SUCCESS          0.0           0.0

projected cost of the full run:
  3.9 s/pick x ~19 picks x 2 pickers x 30 trees = 75 min


## 5. The full run — resumable

Eighty minutes is long enough that a crash on tree twenty-six must not cost the first twenty-five.
Every tree is appended to disk as it finishes and completed trees are skipped on a re-run, so the
cell can be interrupted and restarted.

`MAX_NEIGHBOURS = 2` truncates the scene: a fruit with three close neighbours only ever meets its
two nearest. Section 8 tests whether that truncation changes any label — the reason to look is
that clusters of four to seven fruit do exist in these trees.


In [8]:
POSE_TREES    = [TREES_EVAL[3]]      # trees whose pose stream is kept, for the video
POSE_PICK_CAP = 12                   # and how many picks of each

# Three pickers, not two. "rule" is the shipping configuration and is the one the fidelity gap
# has to be measured on -- the earlier run used the 0.9 threshold, which only ever attempted the
# fruit the outcome model was already confident about, and a gap measured on easy picks does not
# transfer to a plan that attempts everything.
PICKERS = ("rule", "threshold", "policy")

RUN_CSV, LOG_CSV = OUT/"validation_summary.csv", OUT/"validation_picks.csv"

done = set()
if RUN_CSV.exists():
    prev = pd.read_csv(RUN_CSV)
    done = set(map(tuple, prev[["tree", "picker"]].to_numpy()))
    print(f"resuming: {len(done)} tree-picker cells already on disk")
    if done and not any(p == "rule" for _, p in done):
        print("  none of them is the shipping configuration -- this file is from an earlier")
        print("  operating point. Rename it before running, or the run below will skip.")

t0 = time.time()
for tid in TREES_EVAL:
    for picker in PICKERS:
        if (tid, picker) in done:
            continue
        want = picker == "rule" and tid in POSE_TREES
        L, S_, poses = run_tree_mj(tid, picker=picker, log_poses=want)

        pd.DataFrame([S_]).to_csv(RUN_CSV, mode="a", header=not RUN_CSV.exists(), index=False)
        if len(L):
            L.to_csv(LOG_CSV, mode="a", header=not LOG_CSV.exists(), index=False)
        if poses:
            with open(LOGS/f"poses_{tid}_{picker}.jsonl", "w") as f:
                for r in poses:
                    f.write(json.dumps(r) + "\n")
        print(f"  tree {tid:>2} {picker:<10} premium {S_['premium']:>3}  "
              f"attempts {S_['attempts']:>3}  {(time.time()-t0)/60:5.1f} min")

R = pd.read_csv(RUN_CSV).drop_duplicates(["tree", "picker"], keep="last")
P = pd.read_csv(LOG_CSV).drop_duplicates(["tree", "picker", "step"], keep="last")
print(f"\n{len(R)} cells, {len(P)} picks, {(time.time()-t0)/60:.1f} min this session")
print(R.groupby("picker").agg(trees=("tree", "size"), attempts=("attempts", "mean"),
                              premium=("premium", "mean"),
                              seconds=("seconds", "mean")).round(1).to_string())


  tree 20 rule       premium  60  attempts  67    5.5 min
  tree 20 threshold  premium  33  attempts  36    8.7 min
  tree 20 policy     premium  28  attempts  30   11.2 min
  tree 21 rule       premium  52  attempts  59   16.4 min
  tree 21 threshold  premium  34  attempts  36   19.6 min
  tree 21 policy     premium  26  attempts  28   22.1 min
  tree 22 rule       premium  60  attempts  69   28.1 min
  tree 22 threshold  premium  46  attempts  49   32.2 min
  tree 22 policy     premium  46  attempts  49   36.4 min
  tree 23 rule       premium  58  attempts  66   41.7 min
  tree 23 threshold  premium  33  attempts  37   43.7 min
  tree 23 policy     premium  35  attempts  38   45.8 min
  tree 24 rule       premium  42  attempts  50   48.8 min
  tree 24 threshold  premium  20  attempts  22   50.6 min
  tree 24 policy     premium  21  attempts  22   52.6 min
  tree 25 rule       premium  51  attempts  60   56.4 min
  tree 25 threshold  premium  31  attempts  35   58.4 min
  tree 25 poli

## 6. Paired, on the trees themselves

The same paired test section 6 of the evaluation notebook used, but the unit is a tree rather than
a seed, and the outcome came out of the physics rather than out of the model that the policy was
trained against.

Read a confidence interval that spans zero as a bound, not as a failure. The question the write-up
has to answer is how large the difference could be, and the interval answers exactly that.


In [9]:
def paired(a, b, n_boot=20000, seed=0):
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    mean = float(d.mean())
    sd = float(d.std(ddof=1))
    dz = mean/sd if sd > 0 else 0.0
    try:
        from scipy import stats
        t, p = stats.ttest_rel(a, b)
        half = stats.t.ppf(0.975, len(d)-1)*sd/np.sqrt(len(d))
        lo, hi, p = mean-half, mean+half, float(p)
    except Exception:
        rng = np.random.default_rng(seed)
        bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
        lo, hi = np.percentile(bs, [2.5, 97.5])
        p = float(2*min((bs <= 0).mean(), (bs >= 0).mean()))
    return dict(diff=mean, lo=float(lo), hi=float(hi), p=p, d=dz, n=len(d))


W = R.pivot_table(index="tree", columns="picker",
                  values=["premium", "attempts", "lost", "predicted", "touching", "seconds"])

print(f"{len(W)} trees, in the physics\n")
print(f"  {'':<12}{'attempts':>10}{'premium':>10}{'knocked':>10}{'minutes':>10}")
for p in PICKERS:
    if ("premium", p) not in W:
        continue
    print(f"  {p:<12}{W[('attempts', p)].mean():>10.1f}{W[('premium', p)].mean():>10.2f}"
          f"{W[('lost', p)].mean():>10.2f}{W[('seconds', p)].mean()/60:>10.1f}")

base = "rule"
print(f"\n\nagainst the shipping configuration, paired on the tree\n")
for p in PICKERS:
    if p == base or ("premium", p) not in W:
        continue
    res = paired(W[("premium", p)], W[("premium", base)])
    print(f"  {p:<12} {res['diff']:+7.2f} premium  "
          f"[{res['lo']:+.2f}, {res['hi']:+.2f}]  p={res['p']:.4f}  d={res['d']:+.2f}")

print("\n  Per tree, and left per tree. An earlier version of this cell scaled the difference")
print("  to a shift, which needed a time budget the run does not have -- there is no clock")
print("  here, every tree is worked until the plan is done.")

if ("touching", base) in W:
    tp = W[("touching", base)]
    band = pd.cut(tp, [-0.1, 0.5, 2.5, 99], labels=["0 touching", "1-2", "3+"])
    D = pd.DataFrame({"diff": W[("premium", "policy")] - W[("premium", base)], "band": band}) \
        if ("premium", "policy") in W else None
    if D is not None:
        print("\n\npolicy minus rule, by how many propped fruit the tree carried\n")
        print(D.groupby("band", observed=True)["diff"].agg(["count", "mean", "std"])
              .round(3).to_string())


30 trees, in the physics

                attempts   premium   knocked   minutes
  rule              60.7     53.93      5.93      20.0
  threshold         36.9     34.50      1.07      12.0
  policy            32.8     30.90      1.07       8.4


against the shipping configuration, paired on the tree

  threshold     -19.43 premium  [-20.84, -18.02]  p=0.0000  d=-5.15
  policy        -23.03 premium  [-25.27, -20.80]  p=0.0000  d=-3.85

  Per tree, and left per tree. An earlier version of this cell scaled the difference
  to a shift, which needed a time budget the run does not have -- there is no clock
  here, every tree is worked until the plan is done.


policy minus rule, by how many propped fruit the tree carried

      count    mean   std
band                     
3+       30 -23.033  5.98


## 7. Predicted against realised

Two different questions live here and they need to be kept apart.

**Calibration on the picks that were actually chosen.** The layer-1 test split is a random sample;
the fruit a picker selects are not — they are drawn from the high-probability tail. Calibration
holding on a random sample says nothing about whether it holds where the planner spends its time.

**The observation pipeline.** `aipick_env` feeds the model a constant `obs_visible_frac` of 0.36
and a constant 16,000 points, and approximates lean by interpolating the centre distance. The
physics path settles the scene and measures the lean. If prediction and outcome disagree, the
cause could be the policy or it could be that the two paths never saw the same observation, and
the columns below separate those.


In [10]:
# The gap that matters is the one on the picks the shipping configuration makes. Scoring a plan
# that attempts everything against the subset an old threshold would have chosen flatters it:
# the fruit the threshold left alone are the harder ones, and they are in this plan.
seen = sorted(P.picker.unique())
if "rule" not in seen:
    raise RuntimeError(
        f"the pick log carries pickers {seen} and not 'rule'. Earlier revisions of the runner "
        "labelled every rule pick as 'threshold', which let drop_duplicates merge the two -- "
        "delete validation_picks.csv and validation_summary.csv and run section 5 again.")
SHIP = P[P.picker == "rule"].copy()
print(f"calibration on {len(SHIP):,} picks from the shipping configuration "
      f"(of {len(P):,} across all three)\n")

bins = np.array([0, .5, .6, .7, .8, .9, .95, 1.0])
SHIP["hit"] = (SHIP.cls == "SUCCESS").astype(int)
SHIP["bin"] = pd.cut(SHIP.p_success, bins, include_lowest=True)

cal = SHIP.groupby("bin", observed=True).agg(
    n=("hit", "size"), predicted=("p_success", "mean"), observed=("hit", "mean")).round(3)
cal["gap"] = (cal.observed - cal.predicted).round(3)
print(cal.to_string())

pred, obs = SHIP.p_success.mean(), SHIP.hit.mean()
print(f"\noverall  predicted {pred:.3f}   observed {obs:.3f}   gap {obs-pred:+.3f}")
print(f"  the prior run, at three stops and a 0.9 threshold, had "
      f"{PRIOR['predicted']:.3f} against {PRIOR['realised']:.3f}, a gap of {PRIOR['gap']:.3f}")

print("\n\nrealised class mix, shipping configuration\n")
print((SHIP.cls.value_counts(normalize=True)*100).round(2).to_string())

print("\n\nby picker, for comparison\n")
for p in sorted(P.picker.unique()):
    Q = P[P.picker == p]
    h = (Q.cls == "SUCCESS").mean()
    print(f"  {p:<12} n={len(Q):>5}   predicted {Q.p_success.mean():.3f}   "
          f"observed {h:.3f}   gap {h-Q.p_success.mean():+.3f}")

print("\n\nlean: what the planner assumed against what the physics settled\n")
LL = SHIP.dropna(subset=["mj_lean_deg"])
print(f"  n={len(LL)}   env {LL.env_lean_deg.mean():6.2f} deg   "
      f"mujoco {LL.mj_lean_deg.mean():6.2f} deg   "
      f"mean gap {(LL.mj_lean_deg - LL.env_lean_deg).mean():+6.2f} deg")
print(f"  correlation {LL[['env_lean_deg','mj_lean_deg']].corr().iloc[0,1]:.3f}")

LP = LL[LL.propped]
if len(LP):
    print("\n  propped fruit only")
    print(f"  n={len(LP)}   env {LP.env_lean_deg.mean():6.2f}   "
          f"mujoco {LP.mj_lean_deg.mean():6.2f}   "
          f"gap {(LP.mj_lean_deg - LP.env_lean_deg).mean():+6.2f} deg")
    print("\n  The planner under-reports lean where fruit prop each other up. Measuring the pose")
    print("  closed most of the gap; what is left sits here, in the cases where a neighbour is")
    print("  holding the fruit off its own axis.")

# Hand the gap to 06 rather than having it copied by hand.
(OUT/"fidelity_gap.json").write_text(json.dumps(dict(
    picker="rule", n=int(len(SHIP)), predicted=float(pred), observed=float(obs),
    gap=float(obs - pred), stops=int(K_STATIONS), threshold=float(THRESHOLD),
    prior=PRIOR), indent=1))
print(f"\nwritten {OUT/'fidelity_gap.json'}")


calibration on 1,820 picks from the shipping configuration (of 3,912 across all three)

                  n  predicted  observed    gap
bin                                            
(-0.001, 0.5]   377      0.196     0.435  0.239
(0.5, 0.6]       44      0.548     0.591  0.043
(0.6, 0.7]       33      0.647     0.424 -0.223
(0.7, 0.8]       49      0.747     0.592 -0.155
(0.8, 0.9]       95      0.863     0.579 -0.284
(0.9, 0.95]     139      0.928     0.612 -0.316
(0.95, 1.0]    1083      0.986     0.883 -0.103

overall  predicted 0.788   observed 0.730   gap -0.058
  the prior run, at three stops and a 0.9 threshold, had 0.975 against 0.841, a gap of 0.134


realised class mix, shipping configuration

cls
SUCCESS             73.02
NEIGHBOR_KNOCKED    15.88
APPROACH_BLOCKED     5.44
GRASP_FAILED         4.29
DEFECT               1.26
NO_DETACH            0.11


by picker, for comparison

  policy       n=  985   predicted 0.975   observed 0.859   gap -0.116
  rule         n= 1820   

## 8. Is the two-neighbour truncation harmless?

`tree_units` keeps the target and its two nearest neighbours. Everything else within four
diameters is left out of the scene. The neighbour-removal experiment showed that fruit beyond
about 1.6 diameters do not move each other, which makes the four-diameter cut safe — but it says
nothing about dropping a *third* fruit that is genuinely touching.

This re-runs a sample of picks with the cap raised and asks a single question: does any label
change? A parameter, not a restructure.


In [11]:
SAMPLE = P[P.propped].sample(min(40, int(P.propped.sum())), random_state=0)

rows = []
for _, r in SAMPLE.iterrows():
    df = T[T.tree == r.tree]
    out = {}
    for cap in (2, 4):
        MAX_NEIGHBOURS = cap
        got = mj_pick(df, int(r.fruit))
        out[cap] = None if got is None else got["row"]["cls"]
    rows.append(dict(tree=int(r.tree), fruit=int(r.fruit), cap2=out[2], cap4=out[4]))
MAX_NEIGHBOURS = 2

TR = pd.DataFrame(rows)
agree = (TR.cap2 == TR.cap4).mean()
print(f"{len(TR)} propped picks re-run with the cap at 2 and at 4\n")
print(f"  labels agree: {agree*100:.1f}%")
if agree < 1.0:
    print("\n  where they differ")
    print(TR[TR.cap2 != TR.cap4].to_string(index=False))
else:
    print("\n  the truncation does not change any outcome in this sample -- "
          "report it as a checked assumption, not an unchecked one")


40 propped picks re-run with the cap at 2 and at 4

  labels agree: 95.0%

  where they differ
 tree  fruit    cap2             cap4
   43    105 SUCCESS     GRASP_FAILED
   20     69 SUCCESS APPROACH_BLOCKED


## 9. Logs for the renderer

Three files per tree, exactly as agreed. `scene_*.json` is written once and holds everything
static; `events_*.jsonl` is one line per pick and is all the still image needs; `poses_*.jsonl` is
the frame stream and is only kept for the trees named in `POSE_TREES`, because it is the only
large file here.

Nothing in these carries a MuJoCo scene: positions are in the **tree frame**, so
`aipick_replay.py` can build one whole-tree scene of its own and drive it from the ids.


In [12]:
HELPER = ["bin", "hit"]          # section 7's diagnostics, not part of the log


def jsonable(v):
    """Anything pandas hands back, as something json will take."""
    if v is None or (np.isscalar(v) and pd.isna(v)):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return None if np.isnan(v) else float(v)
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    if isinstance(v, (str, int, float)):
        return v
    if hasattr(v, "item"):
        try:
            return jsonable(v.item())
        except Exception:
            pass
    return str(v)                # Interval, Timestamp, Categorical and friends


PW = P.drop(columns=[c for c in HELPER if c in P.columns])

for tid in sorted(set(PW.tree)):
    df = T[T.tree == tid]
    case = CASE_CACHE[tid]
    S = sorted(decode_stations(case), key=lambda r: case["POS"][r][0])
    scene = dict(
        tree=int(tid),
        fruits=[dict(id=int(r.fruit_id), pos=[float(r.x), float(r.y), float(r.z)],
                     dim=float(r.dim), sdir=[float(r.sdir_x), float(r.sdir_y), float(r.sdir_z)],
                     visible=bool(r.vis_any),
                     visible_from=[s for s in E.STATION_X if bool(r.get(f"vis_{s}", False))])
                for _, r in df.iterrows()],
        stations=[dict(idx=i, base=[float(case["POS"][r][0]), float(case["POS"][r][1])],
                       reaches=int(case["COV"][r].sum())) for i, r in enumerate(S)],
        arm=dict(half_x=HALF_X, lift=LIFT, depth=[E.ARM_DEPTH_LO, E.ARM_DEPTH_HI],
                 stand=E.R_STAND),
        frame="tree (y up)")
    (LOGS/f"scene_{tid}.json").write_text(json.dumps(scene, indent=1))

    for picker, g in PW[PW.tree == tid].groupby("picker"):
        with open(LOGS/f"events_{tid}_{picker}.jsonl", "w") as f:
            for _, e in g.sort_values("step").iterrows():
                f.write(json.dumps({k: jsonable(v) for k, v in e.to_dict().items()}) + "\n")

R.to_csv(OUT/"validation_summary.csv", index=False)
PW.to_csv(OUT/"validation_picks.csv", index=False)

print(f"written to {LOGS}")
for f in sorted(LOGS.iterdir())[:12]:
    print(f"  {f.name:<34} {f.stat().st_size/1024:8.1f} KB")
print(f"\n{len(list(LOGS.glob('*')))} files total")


written to c:\aipick\git\runs\fidelity\logs
  events_20_policy.jsonl                 10.6 KB
  events_20_rule.jsonl                   23.8 KB
  events_20_threshold.jsonl              12.9 KB
  events_21_policy.jsonl                 10.0 KB
  events_21_rule.jsonl                   21.1 KB
  events_21_threshold.jsonl              12.9 KB
  events_22_policy.jsonl                 17.4 KB
  events_22_rule.jsonl                   24.6 KB
  events_22_threshold.jsonl              17.5 KB
  events_23_policy.jsonl                 13.5 KB
  events_23_rule.jsonl                   23.6 KB
  events_23_threshold.jsonl              13.3 KB

121 files total
